# df DDPM sampling sweep — diagnostic

Asks one question: is the washed-out synthetic `df` a **sampling artefact**, a
**training-length** shortfall, or a property of the trained model?

Background, all measured locally and recorded in `EXPERIMENT_LOG.md`:

- Memorisation is ruled out. The published synthetic `df` are *farther* from the
  85 real train `df` than genuinely new real `df` (the 14 val `df`) are, in two
  independent spaces, flips included.
- The failure is **not** high-frequency texture. A resolution ladder from 64px
  down to 8px leaves the gap intact (ratio 1.54 → 1.58), so raising the
  generator resolution would not address it.
- What is wrong is coarse: saturation `0.053` against `0.188` for real `df`,
  contrast `0.064` against `0.145`, and mean RGB sitting at ≈`0.5` in every
  channel — the centre of the normalised range. That is a sample regressing
  toward the data mean.

The published set was drawn with DDIM at **50 steps, eta 0.0**. Too few steps,
or a fully deterministic trajectory, can produce exactly that on its own — and
costs nothing to test, because every checkpoint is already on Drive.

Read-only against the fixed split: train and val manifests only, never test.
Descriptive — no thresholds, no pass/fail, nothing is selected or retrained.

In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "d2683ebe09b5124823fada3f39bc30ea201c55bb"
assert len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH", "Pin the reviewed pushed commit before Run all"
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
DIAGNOSTIC_VERSION = "v1_ddpm_sampling_sweep"

# Sweep grid. Widen only deliberately: every extra cell is another full sampling run.
SWEEP_STEPS = (50, 250, 1000)   # 50 reproduces the published set
SWEEP_ETAS = (0.0, 1.0)         # 0.0 reproduces the published set
EPOCH_SWEEP = (60, 80, 100)     # baseline setting across training progress
PRIMARY_EPOCH = 100
N_PER_CONFIG = 128              # ample for colour statistics; not a publication sample
BATCH_SIZE = 64
SEED = 0

SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
DDPM_CHECKPOINT_DIR = SHARED_PROJECT_DIR / "outputs" / "ddpm" / "checkpoints"
JUDGE_CHECKPOINT = SHARED_PROJECT_DIR / "outputs" / "classifier_df585" / "checkpoints" / "C1_seed2" / "best.pt"
DIAGNOSTIC_ROOT = SHARED_PROJECT_DIR / "outputs" / "diagnostics" / DIAGNOSTIC_VERSION
CODE_DIR = Path("/content/ddpm-code")

## Phase 0 — mount Drive, clone the pinned commit, install the generator dependency

In [ ]:
import base64, json, os, subprocess, sys, time
from google.colab import drive, userdata
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project folder: {SHARED_PROJECT_DIR}"
assert DDPM_CHECKPOINT_DIR.is_dir(), f"missing DDPM checkpoints: {DDPM_CHECKPOINT_DIR}"
assert JUDGE_CHECKPOINT.is_file(), f"missing C1 judge checkpoint: {JUDGE_CHECKPOINT}"

token = userdata.get("GH_TOKEN")
assert token and len(token) > 20, "Colab Secret GH_TOKEN with read access to this repo is required"
assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
basic_credential = base64.b64encode(("x-access-token:" + token).encode()).decode()
clone_env = os.environ.copy()
clone_env["GIT_TERMINAL_PROMPT"] = "0"
clone_env["GIT_CONFIG_COUNT"] = "1"
clone_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
clone_env["GIT_CONFIG_VALUE_0"] = "Authorization: Basic " + basic_credential
try:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True, env=clone_env)
finally:
    clone_env["GIT_CONFIG_VALUE_0"] = ""
    token = basic_credential = None
    del token, basic_credential, clone_env
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
remote = subprocess.check_output(["git", "-C", str(CODE_DIR), "remote", "get-url", "origin"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status, "clone must be a clean detached checkout of the pinned commit"
assert "@" not in remote and "x-access-token" not in remote, "clone URL must not embed a credential"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers>=0.27", "pandas>=2.0", "pillow>=9.0"], check=True)

import torch
assert torch.cuda.is_available(), "a GPU runtime is required; Runtime -> Change runtime type -> GPU"
print(json.dumps({"commit": commit, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__}, indent=2))

# The scripts resolve the fixed split through the repository's own config.
os.environ["DDPM_DERM_DATA_DIR"] = str(SHARED_PROJECT_DIR / "data")
os.environ["DDPM_DERM_OUTPUTS_DIR"] = str(SHARED_PROJECT_DIR / "outputs")
env = os.environ.copy()
env["PYTHONPATH"] = str(CODE_DIR / "src")
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONDONTWRITEBYTECODE"] = "1"

available = sorted(p.name for p in DDPM_CHECKPOINT_DIR.glob("run_seed0_epoch*.pt"))
print("checkpoints on Drive:", available)
for epoch in set(EPOCH_SWEEP) | {PRIMARY_EPOCH}:
    assert (DDPM_CHECKPOINT_DIR / f"run_seed0_epoch{epoch:04d}.pt").is_file(), f"missing epoch {epoch}"

## Phase 1 — streaming helper

Every sampling run prints as it goes, so a stall is visible rather than
indistinguishable from slow progress.

In [ ]:
import queue, threading

def run_stream(command, cwd=CODE_DIR, heartbeat=60):
    started = time.monotonic()
    process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines, output_queue = [], queue.Queue()
    def pump():
        for line in process.stdout: output_queue.put(line)
        output_queue.put(None)
    threading.Thread(target=pump, daemon=True).start()
    while True:
        try: line = output_queue.get(timeout=heartbeat)
        except queue.Empty:
            print(f"[subprocess] heartbeat elapsed={time.monotonic()-started:.0f}s alive={process.poll() is None}", flush=True)
            continue
        if line is None: break
        lines.append(line); print(line, end="", flush=True)
    code = process.wait()
    if code: raise subprocess.CalledProcessError(code, command)
    return time.monotonic() - started, "".join(lines)

## Phase 2 — smoke run

Four images at five steps. This exists so an import error, a path mistake or a
checkpoint-shape problem surfaces in well under a minute, instead of after an
hour of sampling. The numbers it prints are meaningless; only that it completes
matters.

In [ ]:
smoke_dir = Path("/content/sweep_smoke")
elapsed, _ = run_stream([
    sys.executable, "-B", "-u", "scripts/ddpm_sampling_sweep.py",
    "--checkpoint-dir", str(DDPM_CHECKPOINT_DIR),
    "--judge-checkpoint", str(JUDGE_CHECKPOINT),
    "--steps", "5", "--etas", "0.0", "--epochs", str(PRIMARY_EPOCH),
    "--primary-epoch", str(PRIMARY_EPOCH),
    "--n", "4", "--batch-size", "4", "--seed", str(SEED),
    "--out", str(smoke_dir / "smoke.json"),
    "--montage", str(smoke_dir / "smoke.png"),
])
print(f"\n[smoke] COMPLETE in {elapsed:.0f}s -- the full sweep below is safe to start")

## Phase 3 — the sweep

`len(SWEEP_STEPS) * len(SWEEP_ETAS)` sampler configurations on epoch
`PRIMARY_EPOCH`, then `EPOCH_SWEEP` at the published baseline setting. Cost is
dominated by the 1000-step cells; expect the whole thing to take a while on a
T4. Results are written to Drive, so an interrupted runtime loses only the
current cell.

In [ ]:
DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
record_path = DIAGNOSTIC_ROOT / "sampling_sweep_record.json"
montage_path = DIAGNOSTIC_ROOT / "sampling_sweep_montage.png"
assert not record_path.exists(), f"a record already exists; move it aside deliberately: {record_path}"

elapsed, _ = run_stream([
    sys.executable, "-B", "-u", "scripts/ddpm_sampling_sweep.py",
    "--checkpoint-dir", str(DDPM_CHECKPOINT_DIR),
    "--judge-checkpoint", str(JUDGE_CHECKPOINT),
    "--steps", *[str(s) for s in SWEEP_STEPS],
    "--etas", *[str(e) for e in SWEEP_ETAS],
    "--epochs", *[str(e) for e in EPOCH_SWEEP],
    "--primary-epoch", str(PRIMARY_EPOCH),
    "--n", str(N_PER_CONFIG), "--batch-size", str(BATCH_SIZE), "--seed", str(SEED),
    "--out", str(record_path), "--montage", str(montage_path),
])
print(f"\n[sweep] COMPLETE in {elapsed/60:.1f} min -> {record_path}")

## Phase 4 — read the answer

The comparison that matters is against **real train df saturation**, with the
published baseline (`steps50_eta0.0`) as the internal control: it should
reproduce the desaturated result already on record. If another configuration
recovers saturation and contrast, the failure was the sampler. If the epoch
sweep is still climbing at 100, the run stopped early. If nothing moves, the
limitation is in the trained model and the next step is a real change.

In [ ]:
record = json.loads(record_path.read_text(encoding="utf-8"))
target = record["reference"]["real_train_df"]
print(f"target (real train df): saturation={target['saturation']:.4f}  contrast={target['contrast_std']:.4f}")
print(f"reference (real val df): embedding_nn_median={record['reference']['real_val_df']['embedding_nn_median']:.4f}\n")

header = f"{'configuration':22} {'saturation':>11} {'contrast':>9} {'embed_nn':>9} {'sat/real':>9}"
for bucket in ("sampler_sweep", "epoch_sweep"):
    print(f"--- {bucket} ---"); print(header)
    for tag, s in sorted(record[bucket].items()):
        print(f"{tag:22} {s['saturation']:11.4f} {s['contrast_std']:9.4f} "
              f"{s['embedding_nn_median']:9.4f} {s['saturation']/target['saturation']:9.2f}")
    print()

best = max(record["sampler_sweep"].items(), key=lambda kv: kv[1]["saturation"])
print(f"highest saturation: {best[0]} at {best[1]['saturation']:.4f} "
      f"({best[1]['saturation']/target['saturation']:.0%} of real df)")
print("published baseline steps50_eta0.0 is the control; it should match the recorded 0.053")

In [ ]:
from IPython.display import Image as ShowImage, display
print("rows:", ", ".join(record["montage"]["rows"]))
display(ShowImage(filename=str(montage_path)))